In [ ]:
def objective_xgb(trial):
    params = {
        "n_estimators": 3000,
        "learning_rate":         trial.suggest_float("learning_rate", 1e-2, 2e-1, log=True),
        "max_depth":             trial.suggest_int("max_depth", 4, 12),
        "min_child_weight":      trial.suggest_float("min_child_weight", 1.0, 10.0),
        "subsample":             trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":      trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":             trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":            trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "objective":             "reg:squarederror",
        "eval_metric":           "rmse",
        "random_state":          42,
        "n_jobs":                -1,
        "tree_method":           "hist",
        "early_stopping_rounds": 50,
    }

    model = XGBRegressor(**params)
    model.fit(X_train, np.log1p(y_train), eval_set=[(X_val, np.log1p(y_val))],verbose=False)
    trial.set_user_attr('best_iteration', model.best_iteration)
    val_pred = np.expm1(model.predict(X_val)).clip(0)
    
    return helper.rmsle(val_pred, y_val)
    
start = time.time()
xgb_study = optuna.create_study(direction="minimize")
xgb_study.optimize(objective_xgb, n_trials=10, show_progress_bar=True)

best_params_xgb = xgb_study.best_params
best_iteration_xgb = xgb_study.best_trial.user_attrs.get('best_iteration', 500)

print("Best RMSLE:", xgb_study.best_value)
print("Best params:",best_params_xgb)
print("Took:", time.time() - start, "seconds")

In [ ]:
import optuna
from xgboost import XGBRegressor
import numpy as np
import time


In [ ]:
def objective_cat(trial):
    params = {
        "iterations": 3000,
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 2e-1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0),
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": 42,
        "early_stopping_rounds": 50,
        "verbose": False,
        "allow_writing_files": False,
    }

    model = CatBoostRegressor(**params)
    model.fit(
        X_train, np.log1p(y_train),
        eval_set=(X_val, np.log1p(y_val)), use_best_model=True,
    )

    trial.set_user_attr('best_iteration', model.best_iteration_)

    preds = np.expm1(model.predict(X_val)).clip(0)
    return helper.rmsle(preds, y_val)

start = time.time()

cat_study = optuna.create_study(direction='minimize')
cat_study.optimize(objective_cat, n_trials=10,show_progress_bar=True)

best_params_cat = cat_study.best_params
best_iteration_cat = cat_study.best_trial.user_attrs.get("best_iteration", 500)

print("Best parameters:", best_params_cat)
print("Best RMSLE:", cat_study.best_value)
print("Took:", time.time() - start, "seconds")

In [ ]:
def objective_lgb(trial):
    # 1. Define the parameter search space
    params = {
        "objective":        "regression",
        "metric":           "rmse",
        "boosting_type":    "gbdt",
        "num_leaves":       trial.suggest_int("num_leaves", 31, 512),
        "learning_rate":    trial.suggest_float("learning_rate", 1e-2, 2e-1, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq":     trial.suggest_int("bagging_freq", 1, 10),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 200),
        "lambda_l1":        trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2":        trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "max_depth":        trial.suggest_categorical("max_depth", [-1, 6, 8, 10, 12, 14]),
        "seed":             42,
        "verbose":          -1,
        "nthread":          -1,
    }
    # 2. Create datasets (train, val) for LightGBM
    lgb_train = lgb.Dataset(X_train, label=np.log1p(y_train))
    lgb_val   = lgb.Dataset(X_val,   label=np.log1p(y_val), reference=lgb_train)
    # 3. Train the model
    model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_val],
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(0)
    ]
) 
    # 4. Evaluate on the validation set
    preds = np.expm1(model.predict(X_val, num_iteration=model.best_iteration)).clip(0)

    # 5. Return metric
    
    # 5. Return the metric score
    trial.set_user_attr('best_iteration', model.best_iteration)
    return helper.rmsle(preds, y_val)


# Create Optuna study to minimize the objective function

start = time.time()
# 1. Create the optuna study and specify appropriate direction
study_lgb = optuna.create_study(direction="minimize")

# 2. Optimize (pay attention to recommended trials; 50 takes too long)
study_lgb.optimize(objective_lgb, n_trials=10)
# 3. Get the best parameters
best_params = study_lgb.best_params
best_lgb_num_boost_round = study_lgb.best_trial.user_attrs.get("best_iteration", 500)
# 4. Print them.
print("Best parameters:", best_params)
print("Best RMSLE:", study_lgb.best_value)
print("Took:", time.time() - start, "seconds"